# Gizli Dirichlet Dağılımı (Latent Dirichlet Allocation) (LDA)

🎯 Bu challenge'ın amacı, **LDA** algoritması (NLP'de Denetimsiz Öğrenme) ile e-posta külliyatı içinde konular bulmaktır.

✉️ İşte 1000'den fazla ***etiketlenmemiş e-posta*** içeren bir koleksiyon. Bunlardan ***konuları çıkarmaya*** çalışalım!

In [1]:
import pandas as pd

url = 'https://d32aokrjazspmn.cloudfront.net/materials/lda_data'

data = pd.read_csv(url, sep=",", header=None)
data.columns = ['text']
data.head()

,text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...


In [2]:
data.shape

(1199, 1)

## (1) Preprocessing 

❓ **Question (Cleaning**) ❓ You're used to it by now... Clean up! Store the cleaned text in a new column "clean_text" of the DataFrame.

In [3]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Küçük harf
    text = text.lower()
    # Email header kalıplarını temizle (From:, Subject: vs)
    text = re.sub(r'(from|subject|to|cc|date|reply-to)\s*:.*', '', text)
    # Email adresleri
    text = re.sub(r'\S+@\S+', '', text)
    # URL'ler
    text = re.sub(r'http\S+|www\S+', '', text)
    # Sayılar ve özel karakterler
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    tokens = text.split()
    # Stopword removal + lemmatization
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

data['clean_text'] = data['text'].astype(str).apply(clean_text)
data[['text', 'clean_text']].head()

[nltk_data] Downloading package stopwords to /home/ubtuna/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/ubtuna/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/ubtuna/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,text,clean_text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...,summary devil pregame show prior hosting pengu...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...,organization national association disorganized...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...,organization university kansa academic computi...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...,organization national association disorganized...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...,organization carleton university line subject ...


## (2) Latent Dirichlet Allocation model

❓ **Soru (Eğitim)** ❓ Potansiyel konuları çıkarmak için bir LDA modeli eğitin

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# CountVectorizer — LDA TF-IDF değil raw count ister
vectorizer = CountVectorizer(
    max_df=0.95,      # çok yaygın kelimeleri at
    min_df=2,         # çok nadir kelimeleri at
    max_features=1000
)

X = vectorizer.fit_transform(data['clean_text'])

# LDA modeli
lda_model = LatentDirichletAllocation(
    n_components=5,      # 5 konu — e-posta dataset için makul başlangıç
    max_iter=10,
    learning_method='online',
    random_state=42
)

lda_model.fit(X)
print("LDA eğitimi tamamlandı!")

LDA eğitimi tamamlandı!


##  (3) Potansiyel konuları görselleştirin

🎁 Potansiyel konularla ilişkili kelimeleri yazdırmak için bir  fonksiyon kodladık.

In [5]:
def print_topics(model, vectorizer):
    for idx, topic in enumerate(model.components_):
        print("Topic %d:" % (idx))
        print([(vectorizer.get_feature_names_out()[i], topic[i])
                        for i in topic.argsort()[:-10 - 1:-1]])

❓ **Soru** ❓ LDA tarafından çıkarılan konuları yazdırın.

In [6]:
print_topics(lda_model, vectorizer)

Topic 0:
[('would', 391.28101215128686), ('church', 309.2326070816648), ('organization', 304.4904800546024), ('people', 287.9625276031933), ('one', 258.4399696156516), ('know', 221.54596170004174), ('university', 220.76118599650792), ('question', 199.66508922087579), ('think', 194.32471782307405), ('thing', 186.68890637436965)]
Topic 1:
[('period', 248.44168592867936), ('pt', 212.71321090257985), ('power', 126.7128197125873), ('chi', 120.53871970716553), ('play', 118.64105341002595), ('bos', 115.62486339006377), ('det', 115.54684061433422), ('van', 111.63342087189464), ('april', 110.1733910662339), ('cal', 105.60780351592217)]
Topic 2:
[('team', 880.6914371879726), ('game', 866.2541514477033), ('player', 536.7578288886712), ('hockey', 521.3265884882694), ('organization', 506.3597925275691), ('year', 378.31864831518754), ('writes', 358.186729328938), ('play', 351.77264132707114), ('university', 347.1628908133182), ('season', 342.6716144501485)]
Topic 3:
[('jesus', 495.93215603728606), (

## (4) Yeni bir metnin belge-konu karışımını tahmin edin

❓ **Soru (Tahmin)** ❓

LDA modeliniz fit edildiğine göre, onu yeni bir metnin konularını tahmin etmek için kullanabilirsiniz.

1. Örneği vektörleştirin
2. Vektörleştirilmiş örnek üzerinde LDA'yı kullanarak konuları tahmin edin

In [7]:
example = ["My team performed poorly last season. Their best player was out injured and only played one game"]

In [8]:
# 1. Örneği vectorize et (aynı vectorizer ile!)
example_vec = vectorizer.transform(example)

# 2. LDA ile konu dağılımını tahmin et
topic_distribution = lda_model.transform(example_vec)

print("Konu dağılımı:")
for i, prob in enumerate(topic_distribution[0]):
    print(f"  Topic {i}: {prob:.4f}")

print(f"\nEn baskın konu: Topic {topic_distribution[0].argmax()}")

Konu dağılımı:
  Topic 0: 0.0224
  Topic 1: 0.0223
  Topic 2: 0.9105
  Topic 3: 0.0224
  Topic 4: 0.0224

En baskın konu: Topic 2


🏁 Tebrikler! LDA'yı hızlı bir şekilde nasıl uygulayacağınızı öğrendiniz.

💾 Not defterinizi `git add/commit/push` yapmayı unutmayın...

🚀 ... ve bir sonraki göreve geçin!